# E04_BallBalance — PPO с обрезкой и GAE

**Урок:** проект-1 «Баланс в 3D» и урок 2.2 курса [cyber-unity-learn](https://rl-cuber-unity-code.com/)
**Что реализуется:** Proximal Policy Optimization для непрерывных действий.
**Ожидаемый результат:** средняя награда **≥ 80** из 100 (шар держится ≥ 16 с из 20).
**Время обучения:** ~12 минут на сид (`QUICK_RUN = True` — около полутора минут).

## Что здесь принципиально нового

A2C (`E04_BallBalance__a2c.ipynb`) использует собранный роллаут **ровно один
раз**: после шага градиента политика уже другая, и формула policy gradient для
этих данных перестаёт быть верной. Взаимодействие со средой при этом дороже
градиентного шага на порядки — данные жалко.

PPO снимает это ограничение. Отношение правдоподобий позволяет честно
пересчитать градиент под изменившуюся политику, а **обрезка** не даёт уйти
далеко от той, которой данные собраны. Благодаря этому по одному роллауту
можно пройти несколько эпох мини-батчами.

Среда, сеть, критик, GAE и бюджет шагов намеренно совпадают с ноутбуком A2C:
иначе сравнение методов было бы сравнением настроек. Различаются только те
параметры, которые **являются** различием методов: длина роллаута и шаг
обучения. A2C делает один шаг градиента на роллаут, поэтому ему нужен короткий
роллаут (16 шагов) и больший шаг; PPO из роллаута в 128 шагов делает
4 эпохи × 4 мини-батча = 16 шагов и потому обходится меньшим.

## Про диапазон действий — самое коварное место примера

Действие приходит в Unity в диапазоне $[-1, 1]$. Но политика выдаёт среднее
в **сырых** единицах, а к диапазону его приводит экспортёр ONNX
преобразованием $\mathrm{clamp}(x, -3, 3)/3$ (контракт ML-Agents,
`docs/04_ONNX_CONTRACT.md`, §4.2).

Значит, **то же самое** преобразование обязано применяться при сборе опыта
в Python. Иначе обучение идёт в одной системе координат, а инференс в Unity —
в другой, награда падает втрое, и причина не видна ни в одном логе. В коде это
`labrl.nets.gaussian_policy.to_env_action`, и в разделе 9 совпадение
проверяется прогоном через настоящий граф ONNX.

## Требования

```powershell
python scripts\build_env.py E04_BallBalance
```


## 1. Окружение

In [1]:
# --- 1. Окружение -------------------------------------------------------
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "python" / "labrl").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("не найден корень репозитория")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "python"))

import numpy as np
import torch
import torch.nn as nn
import mlagents_envs

from labrl.utils.seeding import resolve_device, set_global_seed

SEED = 0
set_global_seed(SEED)
DEVICE = resolve_device("auto")

print(f"python        {platform.python_version()}")
print(f"torch         {torch.__version__}  CUDA {torch.version.cuda}")
print(f"mlagents_envs {mlagents_envs.__version__}")
print(f"устройство    {DEVICE}")
print(f"сид           {SEED}")


python        3.10.11
torch         2.2.1+cu121  CUDA 12.1
mlagents_envs 1.2.0.dev0
устройство    cuda
сид           0


## 2. Конфигурация

`PPOConfig` — свойства **метода**, `OnPolicyTrainConfig` — свойства **процесса
обучения**. Значения общей части совпадают с ноутбуком A2C.

Три параметра, которых у A2C нет: `clip_range` (ширина обрезки), `epochs`
(сколько раз пройти по роллауту) и `minibatch_size`. Именно они превращают
«один шаг по данным» в «несколько шагов, но с поводком».

**Про затухание шага обучения** — см. тот же абзац в ноутбуке A2C: после того
как задача решена, нормированные преимущества состоят из шума критика,
и без затухания политика разрушается (T-13 в `docs/07_TROUBLESHOOTING.md`).


In [2]:
QUICK_RUN = True   # True — быстрая проверка (~1.5 мин); False — полное обучение (~12 мин)

from labrl.algos.ppo import PPOConfig
from labrl.train.onpolicy import OnPolicyTrainConfig
from labrl.utils.schedules import LinearSchedule

TOTAL_STEPS = 10_000 if QUICK_RUN else 100_000

ALGO_CFG = PPOConfig(
    gamma=0.99,              # горизонт дисконтирования
    gae_lambda=0.95,         # компромисс смещение/дисперсия в оценке преимущества
    learning_rate=3e-4,      # шаг Adam, общий для актора и критика
    clip_range=0.2,          # ширина ε обрезки отношения правдоподобий
    epochs=4,                # столько раз проходим по одному роллауту
    minibatch_size=256,
    target_kl=0.0,           # 0 — не прекращать эпохи досрочно; обрезки достаточно
    value_coef=0.5,
    entropy_coef=0.001,
    max_grad_norm=0.5,
    normalize_advantage=True,
)

TRAIN_CFG = OnPolicyTrainConfig(
    total_steps=TOTAL_STEPS,
    rollout_steps=128,       # 128 шагов × 8 арен ≈ 1024 перехода на обновление
    eval_every_steps=max(1, TOTAL_STEPS // 5),
    eval_episodes=20,
    success_threshold=99.0,  # успех = эпизод дожил до MaxStep (максимум ровно 100)
)

# Шаг обучения затухает до нуля к концу ПОЛНОГО бюджета. Расписание, как и ε
# в других примерах, НЕ сжимается вместе с сокращённым прогоном.
LEARNING_RATE = LinearSchedule(start=3e-4, end=0.0, decay_steps=100_000)

print(f"бюджет: {TOTAL_STEPS} шагов сбора, обновление каждые {TRAIN_CFG.rollout_steps}")
print(f"обновлений за прогон: {TOTAL_STEPS // TRAIN_CFG.rollout_steps}, "
      f"градиентных шагов: {TOTAL_STEPS // TRAIN_CFG.rollout_steps * ALGO_CFG.epochs * 4}")


бюджет: 10000 шагов сбора, обновление каждые 128
обновлений за прогон: 78, градиентных шагов: 1248


## 3. Подключение среды

In [3]:
from labrl.envs.registry import get as get_env
from labrl.envs.unity_env import open_unity_env
from labrl.envs.vec_unity_env import VecUnityEnv

ENV_ID = "E04_BallBalance"
BUILD = get_env(ENV_ID).build_path
if not BUILD.is_file():
    raise FileNotFoundError(f"нет билда {BUILD}\nСоберите: python scripts/build_env.py {ENV_ID}")

# time_scale выше ~20 ломает физику PhysX: шаг интегрирования становится
# слишком крупным, и среда перестаёт совпадать с тем, что видно в редакторе.
handle = open_unity_env(ENV_ID, build_path=BUILD, seed=SEED, time_scale=20.0, no_graphics=True)
vec = VecUnityEnv(handle)
obs = vec.reset()
print(handle.describe())
print(f"\nарен (параллельных сред): {vec.num_envs}")


behavior: E04_BallBalance?team=0
наблюдения (1 сенсоров):
  obs_0: shape=(8,)  имя сенсора: 'VectorSensor_size8'
действия: непрерывных 2, дискретные ветки ()

арен (параллельных сред): 8


## 4. Инспекция пространств

`assert` сверяет фактическую среду с `ENV_SPEC.md`. Расхождение обязано падать
здесь, а не проявляться загадочными результатами через час обучения.


In [4]:
OBS_DIM = vec.single_obs_dim
ACTION_DIM = int(vec.action_spec.continuous_size)

print(f"obs_shapes:        {vec.obs_shapes}")
print(f"obs_dim:           {OBS_DIM}")
print(f"непрерывных:       {ACTION_DIM}")
print(f"дискретные ветки:  {tuple(vec.action_spec.discrete_branches)}")
print(f"\nпример наблюдения: {np.array2string(obs[0][0], precision=3)}")
print("  [0:2] наклон платформы (z, x), [2:5] позиция шара, [5:8] скорость шара")

# Значения из строки-контракта ENV_SPEC.md:
# <!-- validator: obs_size=8; discrete_branches=; continuous_size=2; max_step=1000 -->
assert OBS_DIM == 8, f"obs_dim {OBS_DIM} != 8 (ENV_SPEC.md)"
assert ACTION_DIM == 2, f"непрерывных действий {ACTION_DIM} != 2 (ENV_SPEC.md)"
assert len(vec.action_spec.discrete_branches) == 0
print("\nразмерности совпадают с ENV_SPEC.md")


obs_shapes:        [(8,)]
obs_dim:           8
непрерывных:       2
дискретные ветки:  ()

пример наблюдения: [0.055 0.039 0.804 4.    0.174 0.    0.    0.   ]
  [0:2] наклон платформы (z, x), [2:5] позиция шара, [5:8] скорость шара

размерности совпадают с ENV_SPEC.md


## 5. Нейросети

Здесь их **две**, и роли у них разные.

**Актор** выдаёт среднее гауссовой политики: вход `(B, 8)`, выход `(B, 2)`.
Разброс σ хранится отдельным обучаемым параметром `log_std`, не зависящим
от состояния, — так же, как это делает ML-Agents. Причина практическая:
σ(s) позволяет политике «схлопнуть» разброс в удобных состояниях ещё до того,
как она научится в них действовать, и обучение застревает.

**Критик** оценивает ценность состояния: вход `(B, 8)`, выход `(B, 1)`.
В Unity он не уезжает — он существует только ради расчёта преимущества.

Обе сети определяются **здесь** и полностью редактируемы. Активация `tanh`,
а не `relu`: в задачах управления гладкая политика ведёт себя заметно лучше.


In [5]:
from labrl.nets.gaussian_policy import GaussianPolicyNetwork


class MyPolicy(GaussianPolicyNetwork):
    """Актор: obs (B, 8) -> среднее действия (B, 2) в сырых единицах.

    Наследование от GaussianPolicyNetwork даёт готовые log_prob и entropy;
    заменить можно всё тело — алгоритм знает только про эти три метода.
    """

    def __init__(self, obs_dim: int, action_dim: int, hidden: int = 128):
        super().__init__(obs_dim, action_dim, (hidden, hidden), "tanh", log_std_init=-0.3)


class MyCritic(nn.Module):
    """Критик: obs (B, 8) -> V(s) (B, 1)."""

    def __init__(self, obs_dim: int, hidden: int = 128):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, obs):
        return self.body(obs)


policy_net = MyPolicy(OBS_DIM, ACTION_DIM)
value_net = MyCritic(OBS_DIM)

with torch.no_grad():
    probe = torch.zeros(3, OBS_DIM)
    assert policy_net(probe).shape == (3, ACTION_DIM), "актор обязан выдавать (B, action_dim)"
    assert value_net(probe).shape == (3, 1), "критик обязан выдавать (B, 1)"

print(policy_net)
print(f"\nпараметров актора:  {sum(p.numel() for p in policy_net.parameters()):,}")
print(f"параметров критика: {sum(p.numel() for p in value_net.parameters()):,}")
print(f"стартовое σ: {policy_net.clamped_log_std().exp().detach().numpy().round(3)} "
      f"(в единицах Unity ≈ {(policy_net.clamped_log_std().exp() / 3).detach().numpy().round(3)})")


MyPolicy(
  (body): Sequential(
    (0): Linear(in_features=8, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Tanh()
    (4): Linear(in_features=128, out_features=2, bias=True)
  )
)

параметров актора:  17,924
параметров критика: 17,793
стартовое σ: [0.741 0.741] (в единицах Unity ≈ [0.247 0.247])


## 6. Теория и математика метода

Оценка преимущества здесь та же, что у A2C, — GAE:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t), \qquad
\hat A_t = \sum_{l \ge 0} (\gamma\lambda)^l \delta_{t+l}$$

Отличие — в том, как этот сигнал превращается в шаг. Вводится отношение
правдоподобий текущей политики к той, которой собраны данные:

$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\text{старая}}(a_t|s_t)}$$

и обрезанная целевая функция:

$$L^{\text{CLIP}} = \mathbb{E}\Bigl[\min\bigl(r_t \hat A_t,\;
\mathrm{clip}(r_t, 1-\epsilon, 1+\epsilon)\,\hat A_t\bigr)\Bigr]$$

**Смысл минимума.** Если преимущество положительно, вклад ограничен сверху
множителем $1+\epsilon$: повышать вероятность хорошего действия полезно,
но не бесконечно за один заход. Если отрицательно — ограничен снизу
$1-\epsilon$. В обе стороны это означает: шаг, уводящий политику дальше
$\epsilon$, **не даёт дополнительной выгоды**, и градиент по нему обнуляется.

Диагностика метода — `Custom/Clip Fraction` (доля переходов, попавших под
обрезку) и `Custom/Approx KL`. Если доля обрезанных близка к единице, шаг
слишком велик; если она около нуля — обрезка не работает и PPO вырождается
в многократный A2C, то есть в расходимость.

**Про обрыв по времени** — см. тот же абзац в ноутбуке A2C: все успешные
эпизоды этой среды заканчиваются обрывом, и путать его с завершением нельзя.

Ниже — исходный код обновления, тот самый, что выполняется при обучении.


In [6]:
import inspect

from labrl.algos.ppo import PPO

print(inspect.getsource(PPO.update))


    def update(self, batch: RolloutBatch) -> dict[str, float]:
        """Единственное место, где меняются параметры (требование 8.7).

        Несколько эпох мини-батчами по одному роллауту. Данные при этом
        стареют — на то и обрезка: она гасит вклад переходов, для которых
        новая политика ушла от старой дальше чем на ``clip_range``.
        """
        size = len(batch)
        obs = torch.as_tensor(batch.obs, dtype=torch.float32, device=self.device)
        action = torch.as_tensor(batch.action, dtype=torch.float32, device=self.device)
        old_log_prob = torch.as_tensor(batch.log_prob, dtype=torch.float32, device=self.device)
        advantage_all = torch.as_tensor(batch.advantage, dtype=torch.float32, device=self.device)
        returns = torch.as_tensor(batch.returns, dtype=torch.float32, device=self.device)

        stats: dict[str, float] = {}
        clip_fractions: list[float] = []
        epochs_done = 0
        stop = False

        for _ in range(self.cfg.e

## 7. Цикл обучения

Цикл импортируется из `labrl.train.onpolicy` — он **общий** для A2C и PPO:
методы отличаются только правилом обновления, и разный сбор опыта сделал бы
их сравнение бессмысленным.


In [7]:
from labrl.logging.run_dir import create_run_dir
from labrl.logging.tb_logger import TBLogger
from labrl.train.onpolicy import train_on_policy

algo = PPO(policy_net, value_net, ALGO_CFG, device=DEVICE, seed=SEED)
run = create_run_dir(ENV_ID, "ppo", SEED)
run.write_env_info({**handle.env_info(), "num_envs": vec.num_envs, "seed": SEED,
                    "total_steps": TOTAL_STEPS, "quick": QUICK_RUN})
logger = TBLogger(run.tb)

print(f"каталог прогона: {run.root}\n")

result = train_on_policy(
    vec=vec, algo=algo, logger=logger, cfg=TRAIN_CFG, seed=SEED,
    lr_schedule=LEARNING_RATE,
    on_eval=lambda step, r, s: print(f"  шаг {step:>7}: награда {r:>7.2f}, дожили {s:.0%}"),
)

missing = logger.missing_required_tags()
logger.dump_metrics(run.metrics_json, extra={"seed": SEED, "missing_required_tags": list(missing)})
logger.close()

print(f"\nэпизодов: {len(result.episode_returns)}, переходов: {result.transitions}, "
      f"обновлений: {result.updates}")
print(f"время: {result.wall_time:.1f} с")
print(f"обязательные теги схемы 11.2 без данных: {missing or 'нет'}")


каталог прогона: C:\unity-ml-agents-lab\results\E04_BallBalance\ppo\20260815-210211_seed0



  шаг    2000: награда   10.19, дожили 0%


  шаг    4000: награда   13.08, дожили 0%


  шаг    6000: награда   15.20, дожили 0%


  шаг    8000: награда   33.67, дожили 0%


  шаг   10000: награда   86.55, дожили 35%

эпизодов: 438, переходов: 80000, обновлений: 78
время: 73.1 с
обязательные теги схемы 11.2 без данных: нет


## 8. Оценка и кривая обучения

Третий график — σ политики. Он показывает, как метод сам сворачивает разведку:
в начале действия сильно зашумлены, к концу политика становится почти
детерминированной. Если σ падает слишком рано, увеличьте `entropy_coef`.


In [8]:
import matplotlib
matplotlib.use("Agg")   # ноутбук должен исполняться и без графического бэкенда
import matplotlib.pyplot as plt

returns = np.array(result.episode_returns)
window = max(1, len(returns) // 50)
smoothed = np.convolve(returns, np.ones(window) / window, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

axes[0].plot(smoothed)
axes[0].set_title(f"Награда за эпизод (скользящее среднее по {window})")
axes[0].set_xlabel("эпизод"); axes[0].set_ylabel("награда"); axes[0].grid(alpha=0.3)

steps = [s for s, _, _ in result.eval_history]
rewards = [r for _, r, _ in result.eval_history]
axes[1].plot(steps, rewards, marker="o")
axes[1].axhline(80, color="k", ls="--", lw=1, label="критерий приёмки")
axes[1].axhline(100, color="g", ls=":", lw=1, label="максимум")
axes[1].set_title("Eval/Mean Reward (детерминированная политика)")
axes[1].set_xlabel("шаг сбора"); axes[1].grid(alpha=0.3); axes[1].legend()

sigma = algo.policy_net.clamped_log_std().exp().detach().cpu().numpy()
axes[2].bar(["σ (ось Z)", "σ (ось X)"], sigma)
axes[2].set_title("Итоговый разброс политики")
axes[2].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print(f"итоговая оценка: награда {result.last_eval_reward:.2f}, дожили {result.last_eval_success:.0%}")
print(f"лучшая за прогон: {result.best_eval_reward:.2f}")

# Критерий приёмки проверяется только на ПОЛНОМ бюджете. Сокращённый прогон —
# смоук-тест конвейера: измерено, что за 10 000 шагов A2C набирает 8.7 из 100
# при случайном уровне около 7.5, то есть на этом бюджете метод не обязан
# обучиться, и любой порог проверял бы удачу, а не работоспособность.
# Так же ведёт себя scripts/train.py --quick.
THRESHOLD = 80.0   # блок success_criteria конфига

if QUICK_RUN:
    print(f"быстрый прогон: критерий приёмки ({THRESHOLD}) не проверяется — "
          "проверялась работоспособность конвейера")
else:
    assert result.last_eval_reward >= THRESHOLD, (
        f"не достигнут критерий приёмки: {result.last_eval_reward:.2f} < {THRESHOLD}")
    print(f"критерий приёмки достигнут: {result.last_eval_reward:.2f} >= {THRESHOLD}")


итоговая оценка: награда 86.55, дожили 35%
лучшая за прогон: 86.55
быстрый прогон: критерий приёмки (80.0) не проверяется — проверялась работоспособность конвейера


C:\Users\-\AppData\Local\Temp\ipykernel_21796\1830568204.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Экспорт в ONNX и верификация

`policy_module()` возвращает актора; критик в Unity не нужен. Обёртка экспорта
приводит выход к диапазону Unity преобразованием $\mathrm{clamp}(x,-3,3)/3$
и выдаёт его и как `continuous_actions`, и как
`deterministic_continuous_actions`.

Отдельная проверка ниже — сравнение `algo.deterministic_action()` с выходом
**настоящего графа**. Именно это равенство означает, что Unity будет
действовать так же, как оценка в Python; его нарушение — типовая причина
провала требования 10.6.


In [9]:
import onnxruntime as ort

from labrl.export.onnx_export import ActionSpecLite, export_policy_to_onnx
from labrl.export.onnx_verify import verify_onnx_model

# Наблюдения из среды под текущей политикой.
collected, obs_now = [], vec.reset()
while sum(len(c) for c in collected) < 64:
    collected.append(obs_now[0].copy())
    obs_now = vec.step(algo.deterministic_action(obs_now[0])).obs
sample = [np.concatenate(collected, axis=0)[:64]]

spec = ActionSpecLite(continuous_size=ACTION_DIM)
policy = algo.policy_module()
onnx_path = export_policy_to_onnx(policy, spec, vec.obs_shapes, run.onnx / "policy.onnx")

verification = verify_onnx_model(onnx_path, policy, spec, vec.obs_shapes, sample_obs=sample)
print(verification.report())
verification.raise_if_failed()

# Python и граф обязаны давать одно и то же действие.
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
graph_action = session.run(["deterministic_continuous_actions"], {"obs_0": sample[0]})[0]
python_action = algo.deterministic_action(sample[0])
gap = float(np.abs(graph_action - python_action).max())
print(f"\nmax|ONNX - Python| по действию: {gap:.3e}")
assert gap < 1e-5, "оценка в Python и граф расходятся — инференс в Unity будет другим"

model_dest = (REPO_ROOT / "unity" / "MLAgentsLab" / "Assets" / "Envs" / ENV_ID /
              "Models" / f"{ENV_ID}_ppo.onnx")
model_dest.parent.mkdir(parents=True, exist_ok=True)
model_dest.write_bytes(onnx_path.read_bytes())
print(f"модель скопирована в проект Unity: {model_dest.relative_to(REPO_ROOT)}")

vec.close()


OK    структура (onnx.checker)
OK    opset — ожидался 9, в модели 9
OK    имена входов — лишние=[], отсутствуют=[]
OK    имена выходов — лишние=[], отсутствуют=[]
OK    тип выхода continuous_actions — ожидался float32, получен elem_type=1 (float32)
OK    тип выхода deterministic_continuous_actions — ожидался float32, получен elem_type=1 (float32)
OK    прогон onnxruntime, батч 1
OK    прогон onnxruntime, батч 64
OK    константа version_number — ожидалось [3.0], получено [3.0]
OK    константа memory_size — ожидалось [0.0], получено [0.0]
OK    константа continuous_action_output_shape — ожидалось [2.0], получено [2.0]
OK    числовой паритет: deterministic_continuous_actions — max|ONNX-PyTorch| = 1.192e-07, допуск 1e-04
OK    диапазон непрерывных действий — [-0.2614, 0.1281], допустимо [-1, 1] (clamp ±3 / 3)
ИТОГ: ПРОЙДЕНО (13/13)

max|ONNX - Python| по действию: 1.192e-07
модель скопирована в проект Unity: unity\MLAgentsLab\Assets\Envs\E04_BallBalance\Models\E04_BallBalance_ppo.onnx


## 10. Возврат в Unity

### Автоматически (то, чем проверяется приёмка по 10.6)

```powershell
python scripts\check_inference.py --config configs\E04_BallBalance__ppo.yaml --python-reward <оценка из раздела 8>
```

Скрипт соберёт билд с назначенной моделью и `Behavior Type = Inference Only`,
прогонит 20 эпизодов и сравнит награду с Python-оценкой. Порог — 0.8 от неё.

### Вручную (чтобы увидеть глазами)

1. Открыть `unity/MLAgentsLab` в Unity 6000.5.4f1.
2. Открыть `Assets/Envs/E04_BallBalance/Scenes/E04_BallBalance.unity`.
3. Выделить модель в Project → **Tools → RL → Check ONNX Contract**.
4. У агента (компонент на объекте `Platform`): `Model` = импортированная модель,
   `Behavior Type` = `Inference Only`, галочка `Deterministic Inference` —
   Python оценивал именно детерминированную политику.
5. Play: платформа ловит падающий шар и удерживает его мелкими наклонами.

### Если награда в Unity заметно ниже

Типовые причины (10.6) и порядок разбора — `docs/07_TROUBLESHOOTING.md`.
Для непрерывных действий первая версия всегда одна: **приведение диапазона**.
Проверьте, что при сборе опыта действие проходило через `to_env_action`,
а не отправлялось в среду сырым.

### Ручная проверка среды с клавиатуры

`Behavior Type` = `Heuristic Only`, клавиши **←/→** и **↑/↓** наклоняют
платформу. Удержать шар руками возможно, но неудобно — это и есть та задача,
которую решает агент.
